# ColabUMA — UMAによる分子構造最適化

Metaの機械学習ポテンシャル **UMA** で分子の構造最適化を行うノートブックです。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、収束判定を指定する
4. **ランタイム → すべて実行** を押す
5. **2. ファイル選択と事前確認** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
6. 一番下のセルで 3D 表示・エネルギー・収束グラフを確認し、結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。電荷と多重度はフォームの値を使います。ZIPの中の `_opt.log` は最適化の経路をGaussianの出力形式で書いたもので、GaussViewで開くと各ステップの構造とエネルギー変化を確認できます（Gaussianで計算した結果ではありません）。

2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- 最適化は鞍点に収束することがあります。重要な構造は **振動計算** をオンにして確認してください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 小さな分子（数十原子以下）では、DFTを直接使う方が確実な場面が多くあります。
- 遷移状態探索・IRC・周期系（結晶・表面）の最適化には対応していません。
- 結合長・角度・二面角のスキャンは [ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Scan.ipynb) を使ってください。

## ライセンス

- このノートのコード：MIT License
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "loose（0.05）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.05  #@param {type:"number"}
max_steps = 500  #@param {type:"integer"}
run_vibrations = False  #@param {type:"boolean"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。
import html
import os
import re
import shutil
import subprocess

from IPython import get_ipython
from IPython.display import HTML, display


class ColabUMAError(Exception):
    # 利用者に見せるエラー。トレースバックの代わりに日本語の一文と対処法を表示する
    def __init__(self, message, hint=""):
        super().__init__(message)
        self.message = message
        self.hint = hint


def _box(kind, message, hint=""):
    bg, fg, label = {
        "error": ("#fdecea", "#b3261e", "エラー"),
        "warn": ("#fff4e5", "#8a5300", "注意"),
        "ok": ("#e8f5e9", "#1b5e20", "OK"),
    }[kind]
    hint_html = f"<div style='margin-top:4px'>対処法：{html.escape(hint)}</div>" if hint else ""
    display(HTML(
        f"<div style='background:{bg};color:{fg};padding:10px 14px;border-radius:6px;"
        f"margin:6px 0;font-size:14px;line-height:1.6'><b>{label}</b>　{html.escape(message)}{hint_html}</div>"
    ))


def warn(message, hint=""):
    _box("warn", message, hint)


def ok(message):
    _box("ok", message)


def _show_colabuma_error(shell, etype, evalue, tb, tb_offset=None):
    _box("error", evalue.message, evalue.hint)
    return []


get_ipython().set_custom_exc((ColabUMAError,), _show_colabuma_error)

# --- フォームの値の確認 ---
FMAX_PRESETS = {"loose（0.05）": 0.05, "normal（0.02）": 0.02, "tight（0.001）": 0.001}
fmax = fmax_custom if convergence == "数値指定" else FMAX_PRESETS[convergence]
if fmax <= 0:
    raise ColabUMAError("収束判定 fmax は正の数にしてください。", "通常は 0.02 eV/Å 程度です。")
if max_steps < 1:
    raise ColabUMAError("最大ステップ数は1以上にしてください。", "通常は 500 程度です。")
if multiplicity < 1:
    raise ColabUMAError("スピン多重度は1以上の整数にしてください。", "閉殻分子なら1（一重項）です。")
out_prefix = re.sub(r"[^\w\-.]", "_", output_name.strip()) or "result"

# --- GPUの確認（torchを読み込む前に nvidia-smi で調べる） ---
_has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
if not _has_gpu:
    warn("GPUが見つからないため、CPUで計算します。計算はかなり遅くなります。",
         "メニューの「ランタイム → ランタイムのタイプを変更」で「T4 GPU」を選び、もう一度すべて実行してください。")

# --- Hugging Faceのトークン（ノートに保存しない。検証はインストール後に行う） ---
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        _token = userdata.get("HF_TOKEN")
    except Exception:
        _token = None
    if not _token:
        warn("ColabのシークレットにHF_TOKENが登録されていません。",
             "左サイドバーの鍵アイコン（シークレット）で、名前 HF_TOKEN・値にトークンを登録し、"
             "「ノートブックからのアクセス」をオンにしてください。今回は下の欄に貼り付けても続行できます。")
        from getpass import getpass
        _token = getpass("Hugging Faceのアクセストークンを貼り付けてEnter: ").strip()
    if not _token:
        raise ColabUMAError("Hugging Faceのアクセストークンがありません。",
                            "https://huggingface.co/settings/tokens でRead権限のトークンを作り、シークレットにHF_TOKENとして登録してください。")
    os.environ["HF_TOKEN"] = _token
    del _token

# --- ファイル選択 ---
GJF_EXTENSIONS = (".gjf", ".com", ".gjc")
from google.colab import files

_previous = globals().get("upload_name")
print("xyz または gjf（.gjf / .com）ファイルを選んでください。")
if _previous:
    print(f"前回の「{_previous}」をそのまま使うときは Cancel upload を押してください。")
uploaded = files.upload()
if uploaded:
    if len(uploaded) > 1:
        warn("複数のファイルが選ばれたため、最初の1つだけを使います。")
    upload_name, upload_data = next(iter(uploaded.items()))
elif _previous:
    print(f"前回のファイル「{_previous}」を使います。")
else:
    raise ColabUMAError("ファイルがアップロードされませんでした。", "もう一度すべて実行して、xyz または gjf ファイルを選んでください。")

ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
FAIRCHEM_VERSION = "2.25.0"

import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as _pkg_version


def _installed(pkg):
    try:
        return _pkg_version(pkg)
    except PackageNotFoundError:
        return None


# --- ライブラリのインストール（インストール済みならスキップ） ---
if _installed("fairchem-core") != FAIRCHEM_VERSION or not _installed("py3Dmol"):
    _torch_loaded_before = "torch" in sys.modules
    print(f"ライブラリをインストールしています（fairchem-core {FAIRCHEM_VERSION}）。数分かかります…")
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"fairchem-core=={FAIRCHEM_VERSION}", "py3Dmol"],
        capture_output=True, text=True)
    if _r.returncode != 0:
        print("\n".join(_r.stderr.strip().splitlines()[-15:]))
        raise ColabUMAError("ライブラリのインストールに失敗しました（上にpipのメッセージを表示しています）。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    if _torch_loaded_before:
        raise ColabUMAError("インストールは完了しましたが、読み込み済みのライブラリを入れ替えるためセッションの再起動が必要です。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    ok("インストールが完了しました。")
else:
    print(f"fairchem-core {FAIRCHEM_VERSION} はインストール済みです。スキップします。")

# --- Hugging Faceの認証（トークンはノートに保存しない） ---
if not globals().get("_COLABUMA_AUTH_OK"):
    from huggingface_hub import whoami
    from huggingface_hub.utils import GatedRepoError, HfHubHTTPError, RepositoryNotFoundError
    try:
        whoami()
    except Exception as e:
        if getattr(getattr(e, "response", None), "status_code", None) == 401:
            raise ColabUMAError("Hugging Faceのトークンが無効です。",
                                "https://huggingface.co/settings/tokens でトークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    try:
        from huggingface_hub import auth_check
        auth_check("facebook/UMA")
    except ImportError:
        pass  # 古いhuggingface_hubでは確認を省略し、モデル読み込み時に判定する
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except RepositoryNotFoundError:
        raise ColabUMAError("Hugging Faceのトークンが無効です。",
                            "トークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
    except HfHubHTTPError as e:
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    _COLABUMA_AUTH_OK = True
    ok("Hugging Faceの認証とfacebook/UMAへのアクセス権を確認しました。")

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("計算デバイス:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
import gc

from fairchem.core import FAIRChemCalculator, pretrained_mlip
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

_key = (model_name, DEVICE)
if globals().get("_UMA_KEY") != _key:
    globals().pop("_UMA_PREDICTOR", None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{model_name} を読み込んでいます（初回はダウンロードに数分かかります）…")
    try:
        _UMA_PREDICTOR = pretrained_mlip.get_predict_unit(model_name, device=DEVICE)
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except HfHubHTTPError as e:
        _status = getattr(getattr(e, "response", None), "status_code", None)
        if _status in (401, 403):
            raise ColabUMAError("モデルのダウンロードが拒否されました（アクセス権がありません）。",
                                "https://huggingface.co/facebook/UMA でアクセス申請し、トークンが正しいか確認してください。") from None
        raise ColabUMAError(f"モデルをダウンロードできませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    except KeyError:
        raise ColabUMAError(f"モデル名「{model_name}」はこのバージョンのfairchemにありません。",
                            "フォームで別のモデルを選んでください。") from None
    _UMA_KEY = _key
    ok(f"{model_name} を読み込みました。")
else:
    print(f"{model_name} は読み込み済みです。スキップします。")

calc = FAIRChemCalculator(_UMA_PREDICTOR, task_name=task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
import re
import tempfile

import numpy as np
from ase import Atoms
from ase.data import chemical_symbols
from ase.io import read as ase_read

MAX_Z = 83  # UMAのomolタスク（OMol25）が扱える元素は H〜Bi

def check_elements(numbers):
    bad = sorted({chemical_symbols[z] if 0 < z < len(chemical_symbols) else "X"
                  for z in numbers if z < 1 or z > MAX_Z})
    if bad:
        raise ColabUMAError(f"UMAが扱えない元素が含まれています：{', '.join(bad)}",
                            "UMA（omol）が扱えるのは H〜Bi（原子番号1〜83）です。該当する元素を除くか、別の手法を使ってください。")


_FLOAT = re.compile(r"^[-+]?(\d+\.?\d*|\.\d+)([eEdD][-+]?\d+)?$")


def _gjf_cartesian_line(line):
    # 「元素 [凍結フラグ] x y z [ONIOMの層など]」を (元素記号, xyz) にする。読めなければ None
    tok = line.replace(",", " ").split()
    m = re.match(r"[A-Za-z]+|\d+", tok[0])  # "C(Fragment=1)", "C-CA--0.1", "C12" → "C"、"6" → 原子番号
    if not m:
        return None
    label = m.group(0)
    if label.isdigit():
        z = int(label)
        sym = chemical_symbols[z] if 0 < z < len(chemical_symbols) else label
    elif label.lower().startswith("bq") or label.upper() == "X":
        sym = "X"  # ゴースト原子・ダミー原子（後でエラーにする）
    elif label.lower().startswith("tv"):
        sym = "Tv"
    else:
        sym = label[:2].capitalize() if label[:2].capitalize() in chemical_symbols else label[:1].upper()
    nums = tok[1:]
    is_f = [bool(_FLOAT.match(t)) for t in nums]
    if len(nums) >= 4 and re.fullmatch(r"-?\d+", nums[0]) and all(is_f[1:4]) and (len(nums) == 4 or not is_f[4]):
        xyz = nums[1:4]
    elif len(nums) >= 3 and all(is_f[:3]) and (len(nums) == 3 or not is_f[3]):
        xyz = nums[:3]
    else:
        return None
    return tok[0], sym, [float(t.replace("D", "E").replace("d", "e")) for t in xyz]


def read_gjf(path, fname):
    # Gaussianの入力ファイルから構造だけを読む。電荷・多重度は確認用に返すだけで、計算にはフォームの値を使う
    with open(path, encoding="utf-8", errors="replace") as f:
        text = re.split(r"(?im)^\s*--link1--\s*$", f.read())[0]
    lines = [l.split("!")[0].rstrip() for l in text.splitlines()]
    i, n = 0, len(lines)
    while i < n and (not lines[i].strip() or lines[i].lstrip().startswith("%")):
        i += 1
    if i >= n or not lines[i].lstrip().startswith("#"):
        raise ColabUMAError(f"「{fname}」にルートセクション（# で始まる行）が見つかりませんでした。",
                            "Gaussianの入力ファイル（.gjf / .com）か確認してください。")
    for _ in range(2):  # ルートセクションとタイトルを読み飛ばす
        while i < n and lines[i].strip():
            i += 1
        while i < n and not lines[i].strip():
            i += 1
    cm = lines[i].replace(",", " ").split() if i < n else []
    try:
        gjf_cm = (int(cm[0]), int(cm[1]))
    except (IndexError, ValueError):
        raise ColabUMAError(f"「{fname}」の電荷・多重度の行を読み取れませんでした。",
                            "ルートセクションとタイトルの後に空行があり、その次の行が「0 1」のような電荷・多重度になっているか確認してください。") from None
    i += 1
    atom_lines = []
    while i < n and lines[i].strip():
        atom_lines.append(lines[i])
        i += 1
    parsed = [_gjf_cartesian_line(l) for l in atom_lines]
    if atom_lines and all(parsed):
        for raw, sym, _ in parsed:
            if sym == "Tv":
                raise ColabUMAError("gjfに並進ベクトル（Tv）があります。周期系の最適化には対応していません。",
                                    "Tv の行を消して、分子として計算してください。")
            if sym not in chemical_symbols or sym == "X":
                raise ColabUMAError(f"gjfの原子「{raw}」を元素として読み取れませんでした。",
                                    "ゴースト原子（Bq）やダミー原子（X）は使えません。その行を消してください。")
        return Atoms([p[1] for p in parsed], positions=[p[2] for p in parsed]), gjf_cm
    try:  # デカルト座標でなければ Z-matrix とみなして ASE で読む
        atoms = ase_read(path, format="gaussian-in")
        atoms.calc = None
        return atoms, gjf_cm
    except Exception:
        raise ColabUMAError(f"「{fname}」の分子構造を読み取れませんでした。",
                            "GaussViewで開いて保存し直すか、「元素記号 x y z」のデカルト座標で書かれているか確認してください。") from None


# --- 初期構造の読み込み ---
fname, data = upload_name, upload_data
path = os.path.join(tempfile.mkdtemp(), os.path.basename(fname))
with open(path, "wb") as f:
    f.write(data)
if os.path.splitext(fname)[1].lower() in GJF_EXTENSIONS:
    atoms0, gjf_cm = read_gjf(path, fname)
    if gjf_cm != (charge, multiplicity):
        warn(f"gjfに書かれた電荷・多重度（{gjf_cm[0]} {gjf_cm[1]}）とフォームの値（{charge} {multiplicity}）が違います。"
             "計算にはフォームの値を使います。")
    kind = "gjf"
else:
    frames = None
    for fmt in ("extxyz", "xyz"):
        try:
            frames = ase_read(path, index=":", format=fmt)
            break
        except Exception:
            pass
    if not frames:
        raise ColabUMAError(f"「{fname}」をxyzファイルとして読み込めませんでした。",
                            "1行目が原子数、2行目がコメント、3行目以降が「元素記号 x y z」（単位Å）の形式か確認してください。"
                            "Gaussianの入力ファイルなら拡張子を .gjf か .com にしてください。")
    if len(frames) > 1:
        warn(f"「{fname}」には {len(frames)} 個の構造が入っています。先頭の構造だけを使います。")
    atoms0 = frames[0]
    kind = "xyz"
atoms0.set_pbc(False)
atoms0.set_cell(np.zeros((3, 3)))
atoms0.info = {}
source_desc = f"{kind}: {fname}"

# --- 元素・電荷・多重度の確認 ---
numbers = atoms0.numbers
check_elements(numbers)
n_electrons = int(numbers.sum()) - int(charge)
n_unpaired = int(multiplicity) - 1
if n_electrons <= 0 or n_unpaired > n_electrons or (n_electrons - n_unpaired) % 2:
    suggest = "1, 3, …" if n_electrons % 2 == 0 else "2, 4, …"
    raise ColabUMAError(
        f"電荷 {charge:+d}・スピン多重度 {multiplicity} の組み合わせはこの分子では成り立ちません（電子数 {n_electrons}）。",
        f"多重度は 2S+1 です。電子数が偶数なら多重度は奇数、奇数なら偶数になります。この電荷なら多重度は {suggest} が候補です。")
atoms0.info.update({"charge": int(charge), "spin": int(multiplicity)})

ok(f"{atoms0.get_chemical_formula()}（{len(atoms0)} 原子）の初期構造を用意しました。")

In [ ]:
#@title 6. 最適化 { display-mode: "form" }
#@markdown 「収束」のバーは、fmax が最初の値から収束判定までに桁でどこまで近づいたかを示します。途中で fmax が上がるとバーも戻ります。
import time

from ase.optimize import BFGS, FIRE, LBFGS

OPTIMIZERS = {"LBFGS": LBFGS, "BFGS": BFGS, "FIRE": FIRE}
IMAG_THRESHOLD = 20.0  # cm^-1。これより小さい虚振動は数値誤差とみなす


# --- プログレスバー ---
def _bar_row(label, fraction, text):
    fraction = min(max(fraction, 0.0), 1.0)
    color = "#34a853" if fraction >= 1.0 else "#1a73e8"
    return (f"<div style='display:flex;align-items:center;gap:8px;margin:4px 0;font-size:13px'>"
            f"<div style='flex:none;width:5em'>{label}</div>"
            f"<div style='flex:1;max-width:320px;height:14px;background:#dadce0;border-radius:4px;overflow:hidden'>"
            f"<div style='width:{fraction * 100:.1f}%;height:100%;background:{color}'></div></div>"
            f"<div style='flex:none'>{text}</div></div>")


def _convergence_fraction(start, current, target):
    # 最初の fmax から収束判定までに、桁でどこまで近づいたか（0〜1）。fmax が上がればバーも戻る
    if current < target:
        return 1.0
    if start <= target:
        return 0.0
    return np.log(start / current) / np.log(start / target)


def _convergence_row(label, hist):
    f = hist["fmax"]
    return _bar_row("収束", _convergence_fraction(f[0], f[-1], fmax),
                    f"{label}　fmax {f[-1]:.4f} → {fmax} eV/Å　step {len(f) - 1}/{max_steps}")


class _ProgressDisplay:
    # 同じ場所を書き換えて表示する。更新は 0.2 秒に1回まで（force なら必ず）
    def __init__(self):
        self._handle = display(HTML(""), display_id=True)
        self._last = 0.0

    def show(self, rows, force=False):
        now = time.time()
        if force or now - self._last >= 0.2:
            self._handle.update(HTML("".join(rows)))
            self._last = now


def _fmax_of(forces):
    return float(np.sqrt((forces ** 2).sum(axis=1).max()))


def optimize(atoms, label):
    atoms = atoms.copy()
    atoms.calc = calc
    hist = {"energy": [], "fmax": [], "frames": [], "forces": []}
    opt = OPTIMIZERS[optimizer_name](atoms, logfile=None)
    progress = _ProgressDisplay()

    def record():
        forces = atoms.get_forces()
        hist["energy"].append(float(atoms.get_potential_energy()))
        hist["fmax"].append(_fmax_of(forces))
        hist["frames"].append(atoms.positions.copy())
        hist["forces"].append(forces.copy())
        progress.show([_convergence_row(label, hist)])

    opt.attach(record, interval=1)
    t0 = time.time()
    try:
        opt.run(fmax=fmax, steps=max_steps)
    except torch.cuda.OutOfMemoryError:
        raise ColabUMAError("GPUのメモリが足りません。",
                            "モデルを uma-s-1p1 などの小さいものにするか、原子数を減らしてください。") from None
    progress.show([_convergence_row(label, hist)], force=True)
    converged = hist["fmax"][-1] < fmax
    print(f"  {label}: step {opt.nsteps:4d}  E = {hist['energy'][-1]:.6f} eV  fmax = {hist['fmax'][-1]:.4f} eV/Å"
          f"  {'収束' if converged else '未収束'}（{time.time() - t0:.1f} 秒）")
    return {"label": label, "atoms": atoms, "energy": hist["energy"][-1], "fmax": hist["fmax"][-1],
            "steps": opt.nsteps, "converged": converged, "hist": hist}


print(f"{optimizer_name} で最適化します（fmax < {fmax} eV/Å、最大 {max_steps} ステップ）")
best = optimize(atoms0, "構造")
best_atoms = best["atoms"]

# --- 振動計算（オンのときだけ） ---
vib_result = None
if run_vibrations:
    if len(best_atoms) == 1:
        warn("原子が1つなので振動計算は行いません。")
    else:
        from ase.vibrations import Vibrations

        print("振動計算をしています…")
        va = best_atoms.copy()
        va.calc = calc
        vib = Vibrations(va, name=os.path.join(tempfile.mkdtemp(), "vib"), delta=0.01)
        vib.run()
        freqs = vib.get_frequencies()
        # 虚振動は負の値で表す
        signed = np.where(np.abs(freqs.imag) > np.abs(freqs.real), -np.abs(freqs.imag), freqs.real)
        inertia = va.get_moments_of_inertia()
        n_ext = 5 if inertia.min() < 1e-3 * inertia.max() else 6
        internal = np.sort(signed[np.argsort(np.abs(signed))][n_ext:])
        n_imag = int((internal < -IMAG_THRESHOLD).sum())
        vib_result = {"all": np.sort(signed), "internal": internal, "n_imag": n_imag, "n_ext": n_ext}

In [ ]:
#@title 7. 結果表示と保存 { display-mode: "form" }
import datetime
import platform
import zipfile

import ase
import ase.units
import matplotlib.pyplot as plt
import pandas as pd
import py3Dmol

EV_PER_HARTREE = ase.units.Hartree
EV_PER_KCALMOL = ase.units.kcal / ase.units.mol


def xyz_block(symbols, positions, comment):
    lines = [str(len(symbols)), comment]
    lines += [f"{s:<2s} {x:16.10f} {y:16.10f} {z:16.10f}" for s, (x, y, z) in zip(symbols, positions)]
    return "\n".join(lines) + "\n"


def gaussian_log(r, numbers):
    # 最適化の経路を Gaussian の opt 出力と同じ書式で書く（GaussView で経路とエネルギー変化を見るため）
    h = r["hist"]
    to_au = ase.units.Bohr / EV_PER_HARTREE  # eV/Å → Hartree/Bohr
    t = fmax * to_au
    thresholds = (t, t * 2 / 3, t * 4, t * 8 / 3)  # Gaussian の既定値と同じ比率
    dash69 = " " + "-" * 69
    dash67 = " " + "-" * 67
    out = [
        " Entering Gaussian System, Link 0=g16",
        " ******************************************",
        " Output format compatible with Gaussian 16 (Gaussian, Inc.) so that GaussView and other readers can open it.",
        " This file was written by ColabUMA. It is NOT the output of a Gaussian calculation:",
        f" energies and forces come from the UMA machine-learning potential ({model_name}, task={task_name}).",
        " ******************************************",
        " ---------------------------------------------",
        f" #p opt external='ColabUMA {model_name}/{task_name} {optimizer_name}'",
        " ---------------------------------------------",
        " ---------------",
        f" {out_prefix}",
        " ---------------",
        " Symbolic Z-matrix:",
        f" Charge = {int(charge):2d} Multiplicity = {int(multiplicity)}",
    ]
    out += [f" {chemical_symbols[z]:<2s} {x:20.8f}{y:14.8f}{zz:14.8f}" for z, (x, y, zz) in zip(numbers, h["frames"][0])]
    out.append(" ")
    n_frames = len(h["frames"])
    for i, (pos, e, f) in enumerate(zip(h["frames"], h["energy"], h["forces"])):
        out += ["                          Input orientation:", dash69,
                " Center     Atomic      Atomic             Coordinates (Angstroms)",
                " Number     Number       Type             X           Y           Z", dash69]
        out += [f"{k + 1:7d}{z:11d}{0:12d}    {x:12.6f}{y:12.6f}{zz:12.6f}" for k, (z, (x, y, zz)) in enumerate(zip(numbers, pos))]
        out += [dash69, f" SCF Done:  E(UMA) = {e / EV_PER_HARTREE:17.9f}     A.U. after    1 cycles"]
        fa = f * to_au
        out += [dash67, " Center     Atomic                   Forces (Hartrees/Bohr)",
                " Number     Number              X              Y              Z", dash67]
        out += [f"{k + 1:7d}{z:9d}{fx:19.9f}{fy:15.9f}{fz:15.9f}" for k, (z, (fx, fy, fz)) in enumerate(zip(numbers, fa))]
        out += [dash67, f" Cartesian Forces:  Max {np.abs(fa).max():15.9f} RMS {np.sqrt((fa ** 2).mean()):15.9f}",
                " GradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGrad",
                " Berny optimization.",
                " Search for a local minimum.",
                f" Step number{i + 1:4d} out of a maximum of{max_steps + 1:4d}"]
        disp = (h["frames"][i + 1] - pos) / ase.units.Bohr if i + 1 < n_frames else np.zeros_like(pos)
        values = (np.abs(fa).max(), np.sqrt((fa ** 2).mean()), np.abs(disp).max(), np.sqrt((disp ** 2).mean()))
        out.append("         Item               Value     Threshold  Converged?")
        for name, v, th in zip(("Maximum Force", "RMS     Force", "Maximum Displacement", "RMS     Displacement"),
                               values, thresholds):
            out.append(f" {name:<21s}{v:12.6f}{th:13.6f}     {'YES' if v < th else 'NO'}")
        de = (h["energy"][i + 1] - e) / EV_PER_HARTREE if i + 1 < n_frames else 0.0
        out.append(" Predicted change in Energy=" + f"{de:.6E}".replace("E", "D"))
        if i == n_frames - 1 and r["converged"]:
            out += [" Optimization completed.", "    -- Stationary point found."]
        out.append(" GradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGrad")
    if r["converged"]:
        out.append(f" Normal termination of Gaussian 16 at {datetime.datetime.now():%a %b %d %H:%M:%S %Y}.")
    else:
        out += [" Optimization stopped.", f"    -- Number of steps exceeded,  NStep={max_steps}",
                " Error termination request processed by link 9999."]
    return "\n".join(out) + "\n"


symbols = best_atoms.get_chemical_symbols()
E, E_h = best["energy"], best["energy"] / EV_PER_HARTREE
cond = (f"model={model_name} task={task_name} charge={charge} multiplicity={multiplicity} "
        f"optimizer={optimizer_name} fmax_target={fmax}")
opt_xyz = xyz_block(symbols, best_atoms.positions,
                    f"energy_eV={E:.8f} energy_Ha={E_h:.10f} fmax={best['fmax']:.5f} "
                    f"converged={best['converged']} {cond}")

# --- 3D表示 ---
view = py3Dmol.view(width=640, height=420)
view.addModel(opt_xyz, "xyz")
view.setStyle({"stick": {"radius": 0.14}, "sphere": {"scale": 0.24}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

# --- 数値のまとめ ---
summary = pd.DataFrame({"値": [
    f"{best_atoms.get_chemical_formula()}（{len(best_atoms)} 原子）",
    f"{E:.6f} eV",
    f"{E_h:.8f} Hartree",
    f"{best['fmax']:.4f} eV/Å",
    str(best["steps"]),
    "はい" if best["converged"] else "いいえ",
]}, index=["組成", "最終エネルギー", "最終エネルギー", "最終 fmax", "ステップ数", "収束"])
display(summary)
if not best["converged"]:
    warn(f"最大ステップ数（{max_steps}）までに収束しませんでした。結果は途中の構造です。",
         "最大ステップ数を増やすか、fmax を緩めてもう一度実行してください。")

# --- 収束グラフ（Colabは日本語フォントがないため軸ラベルは英語） ---
h = best["hist"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(h["energy"], marker=".", color="#1f77b4")
ax1.set_xlabel("Step")
ax1.set_ylabel("Energy (eV)")
ax1.set_title("Energy")
ax2.semilogy(h["fmax"], marker=".", color="#d62728")
ax2.axhline(fmax, ls="--", color="gray", label=f"fmax = {fmax}")
ax2.set_xlabel("Step")
ax2.set_ylabel("Max force (eV/Å)")
ax2.set_title("Max force")
ax2.legend()
fig.tight_layout()
plt.show()

# --- 振動計算の結果 ---
if vib_result is not None:
    vmin = vib_result["internal"].min()
    print(f"虚振動の本数: {vib_result['n_imag']}（{IMAG_THRESHOLD:.0f} cm⁻¹ 未満の虚振動は数値誤差として除外）")
    print(f"最小振動数: {vmin:.1f} cm⁻¹（負の値は虚振動）")
    if vib_result["n_imag"] > 0:
        warn(f"虚振動が {vib_result['n_imag']} 本あります。極小点ではなく鞍点の可能性があります。",
             "構造を少しずらして再最適化してください。")
    else:
        ok("虚振動はありません。極小点に収束しています。")

# --- 出力ファイルの作成 ---
out_dir = os.path.join(tempfile.mkdtemp(), out_prefix)
os.makedirs(out_dir)
written = {}


def save(name, text):
    p = os.path.join(out_dir, f"{out_prefix}_{name}")
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    written[name] = p


save("opt.xyz", opt_xyz)
save("opt.log", gaussian_log(best, best_atoms.numbers))
save("traj.xyz", "".join(
    xyz_block(symbols, pos, f"step={i} energy_eV={e:.8f} fmax={fm:.5f}")
    for i, (pos, e, fm) in enumerate(zip(h["frames"], h["energy"], h["fmax"]))))


def _v(pkg):
    return _installed(pkg) or "-"


log = [
    "ColabUMA 計算ログ",
    f"日時: {datetime.datetime.now().isoformat(timespec='seconds')}",
    "",
    "[入力条件]",
    f"入力: {source_desc}",
    f"電荷: {charge}", f"スピン多重度: {multiplicity}",
    f"モデル: {model_name}", f"タスク: {task_name}",
    f"最適化アルゴリズム: {optimizer_name}", f"fmax: {fmax} eV/Å", f"最大ステップ数: {max_steps}",
    f"振動計算: {'オン' if run_vibrations else 'オフ'}",
    "",
    "[ソフトウェア]",
    f"Python {platform.python_version()}", f"fairchem-core {_v('fairchem-core')}", f"ase {ase.__version__}",
    f"torch {torch.__version__}", f"numpy {np.__version__}",
    f"デバイス: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}",
    "",
    "[結果]",
    f"E = {E:.8f} eV ({E_h:.10f} Ha)  fmax = {best['fmax']:.5f}  "
    f"steps = {best['steps']}  converged = {best['converged']}",
]
log += ["", f"{'step':>5s} {'energy_eV':>18s} {'fmax_eV/A':>12s}"]
log += [f"{i:5d} {e:18.8f} {fm:12.5f}" for i, (e, fm) in enumerate(zip(h["energy"], h["fmax"]))]
save("log.txt", "\n".join(log) + "\n")

if vib_result is not None:
    freq_lines = [f"# 振動数 (cm^-1)。負の値は虚振動。並進・回転 {vib_result['n_ext']} 個を除いた値",
                  f"# 虚振動の本数（|ν| >= {IMAG_THRESHOLD:.0f} cm^-1）: {vib_result['n_imag']}"]
    freq_lines += [f"{i + 1:4d} {v:12.2f}" for i, v in enumerate(vib_result["internal"])]
    freq_lines += ["", "# 全モード（並進・回転を含む）"] + [f"{i + 1:4d} {v:12.2f}" for i, v in enumerate(vib_result["all"])]
    save("freq.txt", "\n".join(freq_lines) + "\n")

zip_path = os.path.join(os.path.dirname(out_dir), f"{out_prefix}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written.values():
        z.write(p, os.path.basename(p))
print("ZIPの中身:", ", ".join(os.path.basename(p) for p in written.values()))

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("ZIPを保存しました:", zip_path)